# FailureBench: a reproducible research walkthrough

This notebook reads the actual local benchmark artifacts. Its Python code cells were executed when the notebook was generated. Run from the project root. No manuscript or publication claim is made.

Question: why does 90% coverage for each fixed feature-loss pattern differ from protection when a pattern can be chosen for each input?


In [1]:
import csv, json, math
from pathlib import Path
import numpy as np
from failurebench.conformal import conformal_quantile

manifest = json.loads(Path('results/full/manifest.json').read_text(encoding='utf-8'))
print('Run status:', manifest['status'])
print('Predictor fits:', manifest['completed_predictor_fits'])
print('Datasets:', ', '.join(manifest['config']['datasets']))
print('Seeds:', len(manifest['config']['seeds']))
print('Metric cells:', manifest['metric_rows'], '(dependent comparisons, not independent experiments)')


Run status: complete
Predictor fits: 280
Datasets: diabetes, concrete, energy, airfoil, wine_red, synthetic_linear, synthetic_nonlinear
Seeds: 10
Metric cells: 90720 (dependent comparisons, not independent experiments)


## 1. The logical counterexample

Two masks fail on disjoint 10% regions. Each fixed mask covers 90%, but a label-free selector based on the input can choose the failing mask and reduce coverage to 80%. This is a finite-grid population illustration, not an empirical calibration result.


In [2]:
u = (np.arange(10000) + 0.5) / 10000
errors = np.stack((u < 0.1, (u >= 0.1) & (u < 0.2)), axis=1)
selector = np.where(u < 0.1, 0, 1)
print('Fixed-mask coverages:', (~errors).mean(axis=0))
print('Input-dependent selected coverage:', (~errors[np.arange(len(u)), selector]).mean())
print('Simultaneous coverage:', (~errors.any(axis=1)).mean())


Fixed-mask coverages: [0.9 0.9]
Input-dependent selected coverage: 0.8
Simultaneous coverage: 0.8


## 2. Inspect an actual experiment

This uses the energy benchmark, gradient boosting, clean training, and the first prespecified split. Clean is prediction column zero; feature j's removal is column j+1. The complete report averages all ten seeds and shows every dataset.


In [3]:
raw_path = Path('results/full/predictions/energy__histgb__clean__101.npz')
with np.load(raw_path, allow_pickle=False) as saved:
    raw = {key: saved[key] for key in saved.files}
roles = ['train', 'tune', 'cal', 'test']
parts = [raw[role + '_ids'] for role in roles]
print('Split sizes:', {role: len(ids) for role, ids in zip(roles, parts)})
print('Original-row partitions disjoint:', len(np.unique(np.concatenate(parts))) == sum(map(len, parts)))
print('Calibration prediction shape:', raw['cal_pred'].shape)
print('Test prediction shape:', raw['test_pred'].shape)


Split sizes: {'train': 384, 'tune': 153, 'cal': 115, 'test': 116}
Original-row partitions disjoint: True
Calibration prediction shape: (115, 9)
Test prediction shape: (116, 9)


## 3. Calibrate one maximum per original row

The independently fitted mask-specific scale reallocates interval width. The conformal quantile uses calibration labels, but neither scale fitting nor predictor fitting sees those labels. The future-label coverage event is evaluated only after intervals have been frozen.


In [4]:
alpha = 0.1
tune_error = np.abs(raw['tune_pred'] - raw['tune_y'][:, None])
cal_error = np.abs(raw['cal_pred'] - raw['cal_y'][:, None])
test_error = np.abs(raw['test_pred'] - raw['test_y'][:, None])
scales = np.maximum(np.quantile(tune_error, 0.75, axis=0), 0.05)

for name, scale in [('rowmax', np.ones(len(scales))), ('scaled_rowmax', scales)]:
    row_scores = (cal_error / scale).max(axis=1)
    q = conformal_quantile(row_scores, alpha)
    radii = q * scale
    simultaneous = (test_error <= radii).all(axis=1).mean()
    print(name, '| simultaneous coverage:', round(simultaneous, 4),
          '| average normalized width:', round(2 * radii.mean(), 4))


rowmax | simultaneous coverage: 0.9052 | average normalized width: 4.8944
scaled_rowmax | simultaneous coverage: 0.9741 | average normalized width: 1.5659


## 4. Read the full primary comparison

These are equal-weight descriptive averages across the seven datasets for gradient boosting with clean training, all singleton masks, and a 90% target. Repeat-split uncertainty is conditional on the fixed benchmarks and is not population-level evidence.


In [5]:
with Path('results/full/summary.csv').open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
for method in ['mask_matched', 'rowmax', 'scaled_rowmax']:
    chosen = [r for r in rows if r['model'] == 'histgb' and r['training_arm'] == 'clean'
              and r['family'] == 'all' and r['alpha'] == '0.1'
              and r['policy'] == 'simultaneous' and r['method'] == method]
    print(method, '| mean simultaneous coverage:', round(np.mean([float(r['mean_coverage']) for r in chosen]), 4),
          '| mean normalized width:', round(np.mean([float(r['mean_width']) for r in chosen]), 4))


mask_matched | mean simultaneous coverage: 0.6924 | mean normalized width: 1.7649
rowmax | mean simultaneous coverage: 0.906 | mean normalized width: 3.4065
scaled_rowmax | mean simultaneous coverage: 0.9058 | mean normalized width: 2.5294


![Measured endpoint comparison](results/full/endpoint_gap.png)

## 5. Assumptions and the next research step

The theorem assumes exchangeable original rows and a frozen, finite mask family. It averages over calibration and a future original row. It does not imply input-conditional or selected-mask-conditional coverage. Pair deletion is outside the certificate; the report measures that failure boundary. Scaling did not improve width on every dataset.

The max-score proof and normalized conformal construction are established ideas. An original paper would need a reviewed novelty claim, stronger related-method reproductions, and a richer task. Use [the literature audit](literature.md) and [interview notes](interview_notes.md) to identify what you need to understand and extend.
